In [0]:
spark.sql("show tables from syntheaproject.bronze").display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

###### CDF activations for incremental pull by data feed records

In [0]:
patients_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.patients"))
encounters_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.encounters"))
allergies_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.allergies"))
careplans_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.careplans"))
conditions_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.conditions"))
devices_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.devices"))
imaging_studies_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.imaging_studies"))
immunizations_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.immunizations"))
medications_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.medications"))
observations_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.observations"))
organizations_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.organizations"))
payer_transitions_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.payer_transitions"))
payers_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.payers"))
procedures_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.procedures"))
providers_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.providers"))
supplies_cdf = (spark.readStream.option("readChangeFeed", "true").table("syntheaproject.bronze.supplies"))

In [0]:
print(patients_cdf)

###### Incremental Reading from cdf tables and drop cdf focused columns irrelevant to data

In [0]:
from pyspark.sql.functions import *
# Entity - Patients, Providers, Payers, Organizations
patients_df = (patients_cdf.filter(col("_change_type").isin("insert", "update_postimage")).filter(col("is_active") == True).drop("_change_type", "_commit_version", "_commit_timestamp", "sourcefile"))
providers_df = (providers_cdf.filter(col("_change_type").isin("insert", "update_postimage")).filter(col("is_active") == True).drop("_change_type", "_commit_version", "_commit_timestamp", "sourcefile"))
payers_df = (payers_cdf.filter(col("_change_type").isin("insert", "update_postimage")).filter(col("is_active") == True).drop("_change_type", "_commit_version", "_commit_timestamp", "sourcefile"))
organizations_df = (organizations_cdf.filter(col("_change_type").isin("insert", "update_postimage")).filter(col("is_active") == True).drop("_change_type", "_commit_version", "_commit_timestamp", "sourcefile"))

In [0]:
# Event - Encounter, Procedure, Medication, Immunization, Condition, Allergies, Careplans, Observations, Imaging_studies, Supplies, Devices, Payer_Transitions

encounters_df = (encounters_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
procedures_df = (procedures_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
medications_df = (medications_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
immunizations_df = (immunizations_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
conditions_df = (conditions_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
allergies_df = (allergies_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
careplans_df = (careplans_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
observations_df = (observations_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
imaging_studies_df = (imaging_studies_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
supplies_df = (supplies_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
devices_df = (devices_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))
payer_transitions_df = (payer_transitions_cdf.filter(col("_change_type") == "insert").drop("_change_type","_commit_version","_commit_timestamp","sourcefile"))

###### UDF for transformations (rename, trim, hash primary key)

In [0]:
def normalize_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower())
    return df
def add_hash_key(df, hash_col, cols):
    return df.withColumn(hash_col, sha2(concat_ws("||",*[coalesce(col(c).cast("string"),lit("NULL")) for c in cols]), 256))

###### Renaming columns

In [0]:
patients_df = normalize_columns(patients_df)
payers_df = normalize_columns(payers_df)
providers_df = normalize_columns(providers_df)
organizations_df = normalize_columns(organizations_df)

In [0]:
encounters_df = normalize_columns(encounters_df)
procedures_df = normalize_columns(procedures_df)
medications_df = normalize_columns(medications_df)
allergies_df = normalize_columns(allergies_df)
immunizations_df = normalize_columns(immunizations_df)
conditions_df = normalize_columns(conditions_df)
careplans_df = normalize_columns(careplans_df)
observations_df = normalize_columns(observations_df)
supplies_df = normalize_columns(supplies_df)
devices_df = normalize_columns(devices_df)
imaging_studies_df = normalize_columns(imaging_studies_df)
payer_transitions_df = normalize_columns(payer_transitions_df)

In [0]:
patients_df = (patients_df.withColumns({"pat_id": col("id"),"birth_date": col("birthdate"),"death_date": col("deathdate"),"first_name": col("first"),"last_name": col("last"),"zip_code": col("zip"),"bronze_ingestion": greatest(col("ingested_at"), col("updated_at")),"silver_processed": current_timestamp()}))

payers_df= (payers_df.withColumns({"pay_id" : col("id"), "zip_code": col("zip"), "bronze_ingestion": greatest(col("ingested_at"), col("updated_at")),"silver_processed": current_timestamp()}))

providers_df = (providers_df.withColumns({"prov_id" : col("id"), "org_id": col("organization"),"speciality": col("speciality"), "zip_code": col("zip"), "bronze_ingestion": greatest(col("ingested_at"), col("updated_at")),"silver_processed": current_timestamp()}))

organizations_df =  (organizations_df.withColumns({"org_id": col("id"), "zip_code": col("zip"), "bronze_ingestion": greatest(col("ingested_at"), col("updated_at")),"silver_processed": current_timestamp()}))

In [0]:
encounters_df= (encounters_df.withColumns({"enc_id" : col('id'), "pat_id": col("patient"), "org_id": col("organization"), "prov_id":col('provider'), "pay_id": col("payer"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
procedures_df= (procedures_df.withColumns({"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
medications_df= (medications_df.withColumns({"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
careplans_df= (careplans_df.withColumns({"careplan_id" : col('id'), "enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
conditions_df= (conditions_df.withColumns({"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
immunizations_df= (immunizations_df.withColumns({"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
allergies_df= (allergies_df.withColumns({"enc_id" : col('encounter'), "pat_id": col("patient"),"bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
devices_df= (devices_df.withColumns({"dev_id": col("udi"),"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
observations_df= (observations_df.withColumns({"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
payer_transitions_df= (payer_transitions_df.withColumns({"pay_id": col("payer"), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
supplies_df= (supplies_df.withColumns({"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))
imaging_studies_df = (imaging_studies_df.withColumns({"imaging_studies_id": col('id'),"enc_id" : col('encounter'), "pat_id": col("patient"), "bronze_ingestion": col("ingested_at"),"silver_processed": current_timestamp()}))

###### Datatype handeling

In [0]:
patients_df = patients_df.withColumns({"zip_code" : col("zip_code").cast("string")})
payers_df = payers_df.withColumns({"zip_code" : col("zip_code").cast("string")})
providers_df = providers_df.withColumns({"zip_code" : col("zip_code").cast("string")})
organizations_df = organizations_df.withColumns({"zip_code" : col("zip_code").cast("string")})

In [0]:
encounters_df = (encounters_df.withColumns({"code" : col("code").cast("string"), "reason_code": col("reasoncode").cast("string")}))
procedures_df = (procedures_df.withColumns({"code" : col("code").cast("string"), "reason_code": col("reasoncode").cast("string")}))
medications_df = (medications_df.withColumns({"code" : col("code").cast("string"), "reason_code": col("reasoncode").cast("string")}))
allergies_df = (allergies_df.withColumns({"code" : col("code").cast("string")}))
immunizations_df = (immunizations_df.withColumns({"code" : col("code").cast("string")}))
conditions_df = (conditions_df.withColumns({"code" : col("code").cast("string")}))
careplans_df = (careplans_df.withColumns({"code" : col("code").cast("string"), "reason_code": col("reasoncode").cast("string")}))
observations_df = (observations_df.withColumns({"code" : col("code").cast("string")}))
supplies_df = (supplies_df.withColumns({"code" : col("code").cast("string")}))
devices_df = (devices_df.withColumns({"code" : col("code").cast("string")}))
# imaging_studies_df = (imaging_studies_df.withColumns({"zip_code" : col("zip_code").cast("string")}))
# payer_transitions_df = (payer_transitions_df.withColumns({"zip_code" : col("zip_code").cast("string")}))

###### Adding hash pk for tables having composite pk

In [0]:
# Procedure, Medication, Immunization, Condition, Allergies, Observations, Supplies
procedures_df = procedures_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("enc_id"), col("code")), 256)})
medications_df = medications_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("enc_id"), col("code")), 256)})
immunizations_df = immunizations_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("enc_id"), col("code")), 256)})
conditions_df = conditions_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("enc_id"), col("code")), 256)})
allergies_df = allergies_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("enc_id"), col("code")), 256)})
observations_df = observations_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("enc_id"), col("code")), 256)})
supplies_df = supplies_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("enc_id"), col("code")), 256)})
# Payer_Transitions
payer_transitions_df = payer_transitions_df.withColumns({"hash_pk": sha2(concat_ws("-", col("pat_id"), col("pay_id"), col("start_year").cast("string")), 256)})

trim extra spaces from string columns values

In [0]:
def trim_string_columns(df):
    for c, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(c,trim(col(c)))
    return df

In [0]:
patients_df = trim_string_columns(patients_df)
# print('done')
encounters_df = trim_string_columns(encounters_df)
# print('done')
conditions_df = trim_string_columns(conditions_df)
# print('done')
payers_df = trim_string_columns(payers_df)
# print('done')
providers_df = trim_string_columns(providers_df)
organizations_df = trim_string_columns(organizations_df)
observations_df = trim_string_columns(observations_df)
procedures_df = trim_string_columns(procedures_df)
medications_df = trim_string_columns(medications_df)
immunizations_df = trim_string_columns(immunizations_df)
imaging_studies_df = trim_string_columns(imaging_studies_df)
allergies_df = trim_string_columns(allergies_df)
devices_df = trim_string_columns(devices_df)
careplans_df = trim_string_columns(careplans_df)
supplies_df = trim_string_columns(supplies_df)
payer_transitions_df = trim_string_columns(payer_transitions_df)

null validation


In [0]:
patients_df = patients_df.filter(col("pat_id").isNotNull())
payers_df = payers_df.filter(col("pay_id").isNotNull())
providers_df = providers_df.filter(col("prov_id").isNotNull())
organizations_df = organizations_df.filter(col("org_id").isNotNull())

In [0]:
encounters_df = (encounters_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("pay_id").isNotNull()).filter(col("org_id").isNotNull()).filter(col("prov_id").isNotNull()))
conditions_df = (conditions_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("code").isNotNull()))
medications_df = (medications_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("code").isNotNull()))
observations_df = (observations_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("code").isNotNull()))
allergies_df = (allergies_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("code").isNotNull()))
immunizations_df = (immunizations_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("code").isNotNull()))
procedure_df = (procedures_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("code").isNotNull()))
careplans_df = (careplans_df.filter(col("careplan_id").isNotNull()).filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()))
imaging_studies_df = (imaging_studies_df.filter(col('imaging_studies_id').isNotNull()).filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()))
devices_df = (devices_df.filter(col('dev_id').isNotNull()).filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()))
supplies_df = (supplies_df.filter(col("enc_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("code").isNotNull()))
payer_transitions_df = (payer_transitions_df.filter(col("pay_id").isNotNull()).filter(col("pat_id").isNotNull()).filter(col("start_year").isNotNull()))

##### Write to silver

In [0]:
# (encounters_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/encounters").trigger(availableNow=True).toTable("syntheaproject.silver_manual.encounters"))
# (procedures_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/procedures").trigger(availableNow=True).toTable("syntheaproject.silver_manual.procedures"))
# (medications_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/medications").trigger(availableNow=True).toTable("syntheaproject.silver_manual.medications"))
# (immunizations_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/immunization").trigger(availableNow=True).toTable("syntheaproject.silver_manual.immunizations"))
# (conditions_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/conditions").trigger(availableNow=True).toTable("syntheaproject.silver_manual.conditions"))
# (allergies_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/allergies").trigger(availableNow=True).toTable("syntheaproject.silver_manual.allergies"))
# (careplans_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/careplans").trigger(availableNow=True).toTable("syntheaproject.silver_manual.careplan"))
# (observations_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/observations").trigger(availableNow=True).toTable("syntheaproject.silver_manual.observations"))
# (imaging_studies_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/imaging_studies").trigger(availableNow=True).toTable("syntheaproject.silver_manual.imaging_studies"))
# (supplies_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/supplies").trigger(availableNow=True).toTable("syntheaproject.silver_manual.supplies"))
# (devices_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/devices").trigger(availableNow=True).toTable("syntheaproject.silver_manual.devices"))
# (payer_transitions_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/payer_transitions").trigger(availableNow=True).toTable("syntheaproject.silver_manual.payer_transitions"))

Dedupe and upsert Entity tables

In [0]:
patients_batch_df = spark.read.table("syntheaproject.bronze.patients")

patients_batch_df = normalize_columns(patients_batch_df)

patients_batch_df = (patients_batch_df.withColumnsRenamed({"id": "pat_id", "birthdate": "birth_date","deathdate": "death_date","first": "first_name","last": "last_name","zip": "zip_code", "ingested_at" : "bronze_ingestion"}))
patients_batch_df = patients_batch_df.drop("sourcefile")
patients_batch_df = (patients_batch_df.withColumns({"zip_code": col("zip_code").cast("string"),"bronze_ingestion": greatest(col("bronze_ingestion"), col("updated_at")),"silver_processed": current_timestamp()}))
patients_batch_df = trim_string_columns(patients_batch_df)

patients_batch_df = patients_batch_df.filter(col("pat_id").isNotNull())

In [0]:
patients_batch_df.limit(0).write.format("delta").mode("overwrite").saveAsTable("syntheaproject.silver_manual.patients")

In [0]:
def upsert_patients(batch_df, batch_id):
    window_spec = Window.partitionBy("pat_id").orderBy(col("bronze_ingestion").desc())
    batch_df = (batch_df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn"))

    batch_df.createOrReplaceTempView("updates")

    spark.sql("""
        MERGE INTO syntheaproject.silver_manual.patients AS target USING updates AS source
        ON target.pat_id = source.pat_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *
    """)

query = patients_df.writeStream.foreachBatch(upsert_patients).option("checkpointLocation", "/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/patients").trigger(availableNow=True).start()

query.awaitTermination()

In [0]:
providers_batch_df = spark.read.table("syntheaproject.bronze.providers")

providers_batch_df = normalize_columns(providers_batch_df)

providers_batch_df = (providers_batch_df.withColumnsRenamed({"id": "prov_id", "organization": "org_id","speciality": "speciality", "zip": "zip_code", "ingested_at" : "bronze_ingestion"}))
providers_batch_df = providers_batch_df.drop("sourcefile")
providers_batch_df = (providers_batch_df.withColumns({"zip_code": col("zip_code").cast("string"),"bronze_ingestion": greatest(col("bronze_ingestion"), col("updated_at")),"silver_processed": current_timestamp()}))
providers_batch_df = trim_string_columns(providers_batch_df)

providers_batch_df = providers_batch_df.filter(col("prov_id").isNotNull())

providers_batch_df.limit(0).write.format("delta").mode("overwrite").saveAsTable("syntheaproject.silver_manual.providers")

In [0]:
def upsert_providers(batch_df, batch_id):
    window_spec = Window.partitionBy("prov_id").orderBy(col("bronze_ingestion").desc())

    batch_df = (batch_df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn"))

    batch_df.createOrReplaceTempView("updates")

    spark.sql("""MERGE INTO syntheaproject.silver_manual.providers AS target USING updates AS source ON target.prov_id = source.prov_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT * """)

query = providers_df.writeStream.foreachBatch(upsert_providers).option("checkpointLocation", "/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/providers").trigger(availableNow=True).start()

query.awaitTermination()

In [0]:
organizations_batch_df = spark.read.table("syntheaproject.bronze.organizations")

organizations_batch_df = normalize_columns(organizations_batch_df)

organizations_batch_df = (organizations_batch_df.withColumnsRenamed({"id": "org_id", "zip": "zip_code", "ingested_at" : "bronze_ingestion"}))
organizations_batch_df = organizations_batch_df.drop("sourcefile")
organizations_batch_df = (organizations_batch_df.withColumns({"zip_code": col("zip_code").cast("string"),"bronze_ingestion": greatest(col("bronze_ingestion"), col("updated_at")),"silver_processed": current_timestamp()}))
organizations_batch_df = trim_string_columns(organizations_batch_df)

organizations_batch_df = organizations_batch_df.filter(col("org_id").isNotNull())

organizations_batch_df.limit(0).write.format("delta").mode("overwrite").saveAsTable("syntheaproject.silver_manual.organizations")

In [0]:
def upsert_organizations(batch_df, batch_id):
    window_spec = (Window.partitionBy("org_id").orderBy(col("bronze_ingestion").desc()))
    batch_df = (batch_df.withColumn("rn",row_number().over(window_spec)).filter(col("rn") == 1).drop("rn"))
    batch_df.createOrReplaceTempView("updates")

    spark.sql("MERGE INTO syntheaproject.silver_manual.organizations AS target USING updates AS source ON target.org_id = source.org_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")

query = organizations_df.writeStream.foreachBatch(upsert_organizations).option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/organizations").trigger(availableNow=True).start()

query.awaitTermination()

In [0]:
payers_batch_df = spark.read.table("syntheaproject.bronze.payers")

payers_batch_df = normalize_columns(payers_batch_df)

payers_batch_df = (payers_batch_df.withColumnsRenamed({"id": "pay_id", "state_headquartered": "state_headquartered", "zip": "zip_code", "ingested_at" : "bronze_ingestion"}))
payers_batch_df = payers_batch_df.drop("sourcefile")
payers_batch_df = (payers_batch_df.withColumns({"zip_code": col("zip_code").cast("string"),"bronze_ingestion": greatest(col("bronze_ingestion"), col("updated_at")),"silver_processed": current_timestamp()}))
payers_batch_df = trim_string_columns(payers_batch_df)

payers_batch_df = payers_batch_df.filter(col("pay_id").isNotNull())

payers_batch_df.limit(0).write.format("delta").mode("overwrite").saveAsTable("syntheaproject.silver_manual.payers")

In [0]:
def upsert_payers(batch_df, batch_id):
    window_spec = (Window.partitionBy("pay_id").orderBy(col("bronze_ingestion").desc()))
    batch_df = (batch_df.withColumn("rn",row_number().over(window_spec)).filter(col("rn") == 1).drop("rn"))
    batch_df.createOrReplaceTempView("updates")

    spark.sql("MERGE INTO syntheaproject.silver_manual.payers AS target USING updates AS source ON target.pay_id = source.pay_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")

query = payers_df.writeStream.foreachBatch(upsert_payers).option("checkpointLocation","/Volumes/syntheaproject/default/syntheavolume/checkpoints/silver_manual/payers").trigger(availableNow=True).start()

query.awaitTermination()